# N00 - Environment check (Batch A)
Run on the **RTX 5070 PC** (WSL kernel `Python (heirloom)`) and, optionally, on ARCC2 through Open OnDemand.
The cluster GPU check itself is `sbatch scripts/cluster/env_check.sbatch` (compute nodes only).

Pass criteria: CUDA available, `sm_120` in the arch list on the 5070, bf16 matmul works, HF_HOME outside OneDrive, more than 50 GB free.

In [ ]:
import sys, os
from pathlib import Path
REPO = Path(os.environ.get("HEIRLOOM_REPO", Path.cwd().parents[1]))
sys.path.insert(0, str(REPO))
os.chdir(REPO)
print("repo:", REPO)

In [ ]:
import json, subprocess
from heirloom.utils import envinfo, paths
rec_path = envinfo.write_record()
rec = json.loads(rec_path.read_text())
print("saved:", rec_path)
print("exec location:", rec["exec_location"], "| label:", rec["machine_label"])
print("HF_HOME:", rec["hf_home"], "| free:", rec["hf_home_free_disk_gb"], "GB")
print("repo free disk:", rec["repo_free_disk_gb"], "GB | RAM:", rec["ram_gb"], "GB")
rec["packages"]

In [ ]:
import torch
print("torch", torch.__version__, "CUDA build", torch.version.cuda)
assert torch.cuda.is_available(), "CUDA not visible: update the NVIDIA Windows driver (needs CUDA 13 support) and restart WSL"
p = torch.cuda.get_device_properties(0)
print(p.name, round(p.total_memory / 1024**3, 1), "GB", f"sm_{p.major}{p.minor}")
print("arch list:", torch.cuda.get_arch_list())
x = torch.randn(4096, 4096, device="cuda", dtype=torch.bfloat16)
print("bf16 matmul ok:", torch.isfinite(x @ x).all().item())

In [ ]:
import bitsandbytes as bnb, transformers, trl, peft, datasets
print("bitsandbytes", bnb.__version__, "| transformers", transformers.__version__, "| trl", trl.__version__,
      "| peft", peft.__version__, "| datasets", datasets.__version__)
# NF4 sanity: quantize a small layer on the GPU
lin = bnb.nn.Linear4bit(256, 256, quant_type="nf4", compute_dtype=torch.bfloat16).cuda()
print("NF4 layer ok:", lin(torch.randn(2, 256, device="cuda", dtype=torch.bfloat16)).shape)

In [ ]:
assert "OneDrive" not in str(paths.hf_home()), "HF_HOME is inside OneDrive: source ~/.heirloom_env first"
print("environment OK")